In [ ]:
import finlab.data as data
import finlab
import os

finlab_token = os.getenv("FINLAB_API_KEY")
finlab.login(finlab_token)

close = data.get('etl:adj_close')
close = close.loc['2018-01-01':'2024-12-31']

輸入成功!


Your version is 1.2.20, please install a newer version.
Use "pip install finlab==1.2.23" to update the latest version.


In [16]:
import json
path = "../../data/data_info.json"
with open(path, "r") as f:
    data_info = json.load(f)
category = "All"
stock_ids = data_info[category]

crr_close = close[stock_ids]

In [17]:
from collections import Counter
counter = Counter()

for stock_id in crr_close.columns:
    # 取出該股票對應的價格 Series
    stock_series = crr_close[stock_id]
    
    # 若要忽略 NaN，可以先用 dropna() 避免計算時產生錯誤或偏差
    # 這樣會使得索引順序可能不連續，因此改用數字 iloc 來走訪
    stock_series = stock_series.dropna()
    
    # 走訪這個股票在剩餘有效資料中，能夠比較「當天 vs 30天後」的最大範圍
    for i in range(len(stock_series) - 30):
        # 分別取出當天和 30 天後的價格
        current_price = stock_series.iloc[i]
        future_prices = stock_series.iloc[i: i + 30]
        max_price = max(future_prices)
        
        
        
        # 檢查是否大於 1.5 倍
        if max_price / current_price > 1.5:
            counter[stock_id] += 1
            break

print(f"{len(counter)}/{len(stock_ids)}")         

517/819
